## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import contextily as ctx
import xyzservices as xyz
import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona
fiona.drvsupport.supported_drivers["KML"] = "rw"

# for parsing HTML inside the Description field
from bs4 import BeautifulSoup

# for TIFF files
import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats
from rasterio.features import geometry_mask

In [3]:
from gridsample.utils import create_ids, save_shapefiles

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks"

## Load raw shapes and process

### Dhar

In [5]:
# Load dhar181224
raw_dhar_gdf = gpd.read_file(
    RAW_DATA_DIR / "solar_park_shapefiles" / "dhar181224" / "doc.kml",
    driver="KML"
)

In [6]:
# remove z-dimension
raw_dhar_gdf.geometry = raw_dhar_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons and only keep the polygon
raw_dhar_gdf = raw_dhar_gdf.explode(index_parts=False)
raw_dhar_gdf = raw_dhar_gdf[raw_dhar_gdf.geometry.type == "Polygon"]

# remove useless Description column
dhar_gdf = raw_dhar_gdf.drop(columns="Description")

In [7]:
# drop large green shapes (open .KMZ on Google Earth to see)
dhar_gdf = dhar_gdf[dhar_gdf["Name"] != ""]

In [8]:
# clean up Name so we can separate the villages (string names) from the areas (numbers only)
dhar_gdf["cleaned_name"] = [
    value[0].strip() for value in dhar_gdf["Name"].str.split("/")
]
dhar_gdf["cleaned_name"] = [
    value[0].strip() for value in dhar_gdf["cleaned_name"].str.split(",")
]
# manual clean
dhar_gdf.loc[dhar_gdf["Name"] == "2829Z1", "cleaned_name"] = "2829"

In [ ]:
# ISOLATE AREA ONLY - select rows that have a number as their Name
dhar_yellow_shapes_gdf = dhar_gdf[dhar_gdf["cleaned_name"].str.isnumeric()]
dhar_yellow_shapes_gdf.plot()

In [ ]:
# ISOLATE VILLAGES ONLY - select rows that have a string as their Name
dhar_village_shapes_gdf = dhar_gdf[~dhar_gdf["cleaned_name"].str.isnumeric()]
dhar_village_shapes_gdf = dhar_village_shapes_gdf.drop(columns="cleaned_name")
dhar_village_shapes_gdf = dhar_village_shapes_gdf.rename(
    columns={"Name": "village_name"}
)
dhar_village_shapes_gdf.plot(column="village_name")

In [ ]:
# add village names to areas
dhar_processed_areas_gdf = dhar_yellow_shapes_gdf.sjoin(
    dhar_village_shapes_gdf, how="left", predicate="intersects"
).drop(columns="index_right")
dhar_processed_areas_gdf.plot(column="village_name")
print("Missing village name:", dhar_processed_areas_gdf["village_name"].isnull().sum())
print("Has village name:", dhar_processed_areas_gdf["village_name"].notnull().sum())

In [12]:
# save to file
save_shapefiles(
    dhar_processed_areas_gdf,
    PROCESSED_DATA_DIR / "processed_areas",
    "dhar_processed_all",
    formats=["kml", "parquet"],
)

### Sagar

In [15]:
# ### Unzip KMZ to get KML - USE GOOGLE EARTH TO DO THIS CORRECTLY
# import zipfile

# # Extract the .kml file from the .kmz file
# with zipfile.ZipFile(RAW_DATA_DIR / "solar_park_shapefiles" / "sagar220724.kmz", 'r') as kmz:
#     kmz.extractall(RAW_DATA_DIR / "solar_park_shapefiles" / "sagar220724" )

In [16]:
gdfs = []

for filename in ["sagar_khamkuwa", "sagar_mokalpur", "sagar_tekapar"]:
    gdf = gpd.read_file(
        RAW_DATA_DIR / "solar_park_shapefiles" / "Google Earth" / f"{filename}.kml",
        driver="KML",
    )
    gdf["source"] = filename
    gdfs.append(gdf)

raw_sagar_gdf = pd.concat(gdfs, ignore_index=True)

In [ ]:
# remove z-dimension
raw_sagar_gdf.geometry = raw_sagar_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons (each one just contains one polygon anyway)
raw_sagar_gdf = raw_sagar_gdf.explode(column='geometry')
raw_sagar_gdf.plot(column="source", legend=True)

In [18]:
# Parse description variables

def description_parser(html_content):
    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the inner table containing the attributes
    inner_table = soup.find_all('table')[1]

    # Extract rows from the inner table
    rows = inner_table.find_all('tr')

    # Create a dictionary to store attributes and their values
    data = {}
    for row in rows:
        cols = row.find_all('td')
        if len(cols) == 2:
            key = cols[0].text.strip()
            value = cols[1].text.strip()
            data[key] = value

    return pd.DataFrame([data])

In [19]:
# make dataframe of variables
data = [description_parser(raw_sagar_gdf["Description"].values[i]) for i in range(len(raw_sagar_gdf))]
df = pd.concat(data)
df.set_index(raw_sagar_gdf.index, inplace=True)

In [20]:
# merge with shapes
raw_sagar_gdf.drop(columns=["Name", "Description"], inplace=True)
sagar_gdf = raw_sagar_gdf.merge(df, left_index=True, right_index=True)

In [ ]:
sagar_gdf.plot(column="PAR_TYPE")

In [22]:
save_shapefiles(
    sagar_gdf,
    PROCESSED_DATA_DIR / "processed_areas",
    "sagar_processed_all",
    formats=["kml", "parquet"],
)

In [23]:
# # get unique values for all columns
# for col in sagar_gdf.columns:
#     print(f"Column: {col}")
#     print(sagar_gdf[col].unique())
#     print("\n")

## Load processed shapes

In [24]:
dhar_processed_areas_gdf = gpd.read_parquet(PROCESSED_DATA_DIR / "processed_areas" / "dhar_processed_all.parquet")
dhar_processed_areas_gdf["source"] = "dhar"

In [25]:
dhar_processed_areas_gdf["parcel_id"] = "DHAR_" + dhar_processed_areas_gdf["Name"]

# Function to add suffix to duplicates
def add_suffix_to_duplicates(series):
    counts = {}
    result = []
    for item in series:
        if item in counts:
            counts[item] += 1
            result.append(f"{item}_{counts[item]}")
        else:
            counts[item] = 0
            result.append(item)
    return result

# Apply the function to the parcel_id column
dhar_processed_areas_gdf["parcel_id"] = add_suffix_to_duplicates(dhar_processed_areas_gdf["parcel_id"])

In [26]:
sagar_gdf = gpd.read_parquet(PROCESSED_DATA_DIR / "processed_areas" / "sagar_processed_all.parquet")

In [27]:
sagar_gdf["parcel_id"] = "SAGAR_" + sagar_gdf["UNQID"]

In [28]:
# filter to only the "PA" PAR_TYPE (since it looks like the barren land)
sagar_gdf = sagar_gdf[sagar_gdf["PAR_TYPE"] == "PA"]

In [29]:
gdf = pd.concat([dhar_processed_areas_gdf, sagar_gdf], ignore_index=True)

In [ ]:
gdf.plot()

In [31]:
gdf = gdf[["source", "parcel_id", "geometry"]]

In [ ]:
gdf["parcel_area"] = gdf["geometry"].to_crs("24378").area

## External data sources

### 1. Buildings

In [33]:
from s2cell.s2cell import lat_lon_to_cell_id
import boto3

#### Download rooftop data

Get the ID of the level 6 S2 Cell that this area sits inside

In [34]:
s2_ids = []

for index, row in gdf.iterrows():
    lat = row["geometry"].centroid.y
    lon = row["geometry"].centroid.x
    s2_cell_id = lat_lon_to_cell_id(lat=lat, lon=lon, level=6)
        
    s2_ids.append(s2_cell_id)

s2_ids = list(set(s2_ids))


Download closest S2 cell shapefile from https://beta.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/

In [ ]:
for s2_cell_id in s2_ids:
    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"

    if s2_rooftops_path.exists():
        print("File already exists")
    else:
        s3 = boto3.client("s3", endpoint_url="https://data.source.coop")
        s3.download_file(
            "vida",
            f"google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/{s2_cell_id}.parquet",
            str(s2_rooftops_path),
        )
        print("File downloaded.")

#### Load and process rooftop data

In [36]:
rooftop_gdf_list = []
for s2_cell_id in s2_ids:

    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"
    rooftop_gdf = gpd.read_parquet(s2_rooftops_path)
    rooftop_gdf_list.append(rooftop_gdf)

rooftop_gdf = pd.concat(rooftop_gdf_list, ignore_index=True)

In [37]:
rooftop_gdf = rooftop_gdf.drop(columns=["boundary_id", "s2_id", "geohash", "bbox", "country_iso"])

In [38]:
rooftop_gdf["rooftop_id"] = create_ids(len(rooftop_gdf), f"ROOFTOP_S2_{s2_cell_id}_")

In [ ]:
rooftop_gdf

#### Match rooftops to colonies

In [ ]:
subset_rooftops_gdf = rooftop_gdf.sjoin(gdf, how="inner", predicate="intersects").drop(columns=["index_right"])
subset_rooftops_gdf

#### Produce plots and save to file

In [41]:
for source in gdf["source"].unique():
    folder_path = PROCESSED_DATA_DIR / source
    folder_path.mkdir(parents=True, exist_ok=True)

    area_rooftops = subset_rooftops_gdf[subset_rooftops_gdf["source"] == source]
    ax = gdf[gdf["source"] == source].plot(color="orange", alpha=0.2, figsize=(10, 15))

    # plot buildings
    area_rooftops.plot(ax=ax)
    # ctx.add_basemap(ax, crs=subset_rooftops_gdf.crs.to_string(), source=xyz.TileProvider.from_qms("Google Satellite Hybrid"))
    ax.set_axis_off()
    plt.title(f"{source} Rooftops")
    
    plt.savefig(folder_path / "rooftops.png", bbox_inches="tight")
    plt.close()

    save_shapefiles(
        area_rooftops,
        folder_path,
        f"raw {source} rooftops",
        formats=["parquet", "kml"],
    )

    ## Get rooftop stats
    area_rooftops["area_in_meters"].hist(bins=50)
    plt.xlabel("Rooftop area (m2)")
    plt.ylabel("Number of rooftops")
    plt.title(f"{source} Rooftop Area Distribution")
    plt.savefig(folder_path / "rooftop_area_distribution.png")
    plt.close()

#### Pivot to get area-per-row dataset

In [42]:
rooftop_totals_df = (
    subset_rooftops_gdf.groupby("parcel_id")
    .agg(
        total_rooftop_area_m2=("area_in_meters", "sum"),
        total_rooftop_count=("parcel_id", "size"),
    )
    .reset_index()
)

In [68]:
gdf_with_stats = gdf.merge(rooftop_totals_df, on="parcel_id")

In [69]:
gdf_with_stats["perc_area_covered_by_buildings"] = gdf_with_stats["total_rooftop_area_m2"] / gdf_with_stats["parcel_area"] * 100

### 2. Solar datasets

In [70]:
solar_data_folderpath = (
    RAW_DATA_DIR
    / "solar_atlas"
    / "India_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF"
)

#### Averages

In [71]:
solar_irradiation_filenames = [
    "GTI.tif",
    "DIF.tif",
    "DNI.tif",
    "GHI.tif",
    "PVOUT.tif",
    "TEMP.tif",
    "OPTA.tif",
    
]
solar_irradiation_column_names = [
    "Average Yearly Optimum Tilt Irradiation, GTI (kWh/m2)",
    "Average Yearly Diffuse Irradiation, DIF (kWh/m2)",
    "Average Yearly Direct Normal Irradiation, DNI (kWh/m2)",
    "Average Yearly Horizontal Irradiation, GHI (kWh/m2)",
    "Average Yearly Potential PV Output, PVOUT (kWh/kWp)",
    "Average Yearly Temperature (C)",
    "Average Yearly Optimum Angle (deg)",
]

In [72]:
solar_stats = []
for filename, column_name in zip(
    solar_irradiation_filenames, solar_irradiation_column_names
):
    raster = rasterio.open(solar_data_folderpath / filename)
    # Prep data
    array = raster.read(1)
    affine = raster.transform
    # get per-rooftop irradiation
    solar_stats = zonal_stats(
        gdf,
        array,
        affine=affine,
        nodata=raster.nodata,
        stats=["mean"],
        all_touched=True,
    )
    # add to rooftops dataset
    gdf[column_name] = [d["mean"] for d in solar_stats]
    # get per-colony stat
    irradiation_colony_totals = gdf.groupby("parcel_id")[
        column_name
    ].mean()

    # add to dataset
    gdf_with_stats[column_name] = gdf_with_stats["parcel_id"].map(irradiation_colony_totals)

#### Total yearly GTI, DIF, DNI, GHI

In [73]:
solar_irradiation_filenames = [
    "GTI.tif",
    "DIF.tif",
    "DNI.tif",
    "GHI.tif",
]
solar_irradiation_column_names = [
    "Total Yearly Optimum Tilt Irradiation, GTI (kWh)",
    "Total Yearly Diffuse Irradiation, DIF (kWh)",
    "Total Yearly Direct Normal Irradiation, DNI (kWh)",
    "Total Yearly Horizontal Irradiation, GHI (kWh)",
]

In [74]:
solar_stats = []
for filename, column_name in zip(
    solar_irradiation_filenames, solar_irradiation_column_names
):
    raster = rasterio.open(solar_data_folderpath / filename)
    # Prep data
    array = raster.read(1)
    affine = raster.transform
    # get per-rooftop irradiation
    solar_stats = zonal_stats(
        gdf,
        array,
        affine=affine,
        nodata=raster.nodata,
        stats=["sum"],
        all_touched=True,
    )
    # add to rooftops dataset
    gdf[column_name] = [d["sum"] for d in solar_stats]
    # get per-colony stat
    irradiation_colony_totals = gdf.groupby("parcel_id")[
        column_name
    ].sum()

    # add to dataset
    gdf_with_stats[column_name] = gdf_with_stats["parcel_id"].map(irradiation_colony_totals)

In [ ]:
gdf_with_stats

### 3. Landcover (TBD)

In [ ]:
path = "../data/00_raw/landcover/30N_070E_2020.tif"
src = rasterio.open(path)
print("Profile:", src.profile)
print("Number of bands:", src.count)
print("Band indexes:", src.indexes)
show(src, cmap='terrain')

# Read the entire dataset
data = src.read()
print("Data shape:", data.shape)
# get a list of the values in the first band
data.flatten()

# Don't forget to close the dataset when done
src.close()

### 4. Slope

#### Load slope data

In [129]:
path = RAW_DATA_DIR / "elevation" / "cdnf44a.tif"
dem_dataset = rasterio.open(path)

In [130]:
dem = dem_dataset.read(1)
affine = dem_dataset.transform

In [ ]:
show(dem_dataset, cmap='gray')

#### Calculate gradients

In [133]:
# Example 2D array of elevation, and constant gridsize
elev = dem
cellsize = 30

px, py = np.gradient(elev, cellsize)
slope = np.sqrt(px ** 2 + py ** 2)
slope_deg = np.degrees(np.arctan(slope))

In [143]:
# Function to calculate the percentage of area with slope > 7 degrees
def calculate_slope_percentage(shape, slope, affine):
    mask = geometry_mask([shape], transform=affine, invert=True, out_shape=slope.shape)
    slope_in_shape = slope[mask]
    total_area = np.sum(mask)
    steep_area = np.sum(slope_in_shape > 7)
    return (steep_area / total_area) * 100

In [147]:
# Apply the function to each shape in the GeoDataFrame
gdf['slope_higher_than_7deg'] = gdf.geometry.apply(lambda shape: calculate_slope_percentage(shape, slope_deg, affine))

In [ ]:
gdf

#### Get stats

In [139]:
test = zonal_stats(
    gdf,
    slope_deg,
    affine=affine,
    stats=["min", "max", "mean", "std"],
    all_touched=True,
)

### 5. Ground water

In [76]:
water_2017_sod_gdf = gpd.read_file(
    RAW_DATA_DIR / "water" / "Ground Water Resource SOD AU 2017.geojson"
)

In [77]:
water_2017_sod_gdf = water_2017_sod_gdf[water_2017_sod_gdf["State_Name_2017"] == "MP"]

In [ ]:
water_2017_sod_gdf.plot(column="CLASS", legend=True)

In [89]:
gdf_with_stats_2 = gdf_with_stats.sjoin(
    water_2017_sod_gdf,
    how="left",
    rsuffix="groundwater",
    predicate="intersects",
)

In [90]:
major_aquifers_gdf = gpd.read_file(
    RAW_DATA_DIR / "water" / "Major Aquifers.geojson"
)

In [91]:
gdf_with_stats_3 = gdf_with_stats_2.sjoin(
    major_aquifers_gdf,
    how="left",
    rsuffix="aquifer",
    predicate="intersects",
)

## Distance from water bodies

## Distance from roads

## Save to File

In [ ]:
len(gdf_with_stats_3.columns)

In [92]:
save_shapefiles(
    gdf_with_stats_3,
    PROCESSED_DATA_DIR,
    "land_parcels_with_stats",
    formats=["parquet", "kml", "csv"],
)